### Ingestion del archivo "language.csv"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

#### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark

In [0]:
from pyspark.sql.types import StructField, StringType, IntegerType, StructType

In [0]:
languages_schema = StructType(fields=[
    StructField("languageId", IntegerType(), False),
    StructField("languageCode", StringType(), True),
    StructField("languageName", StringType(), True)
])

In [0]:
language_df = spark.read \
                .schema(languages_schema) \
                .option("header", "true") \
                .csv(f"{bronze_folder_path}/{v_file_date}/language.csv")

In [0]:
display(language_df)

#### Paso 2 - Seleccionar solo las columnas "requeridas"

In [0]:
from pyspark.sql.functions import col

In [0]:
language_selected_df = language_df.select(col("languageId"), col("languageName"))

#### Paso 3 - Cambiar el nombre de las columnas segun lo "requerido"

In [0]:
language_renamed_df = language_selected_df.withColumnsRenamed({"languageId": "language_id", "languageName": "language_name"})

#### Paso 4 - Agregar la columna "ingestion_date" y "environment" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# language_final_df = add_ingestion_date(language_renamed_df) \
#                     .withColumn("environment", lit("production"))

language_final_df = add_ingestion_date(language_renamed_df) \
                    .withColumn("environment", lit(v_environment)) \
                    .withColumn("file_date", lit(v_file_date))

In [0]:
display(language_final_df)

#### Paso 5 - Escribir datos en el datalake en formato "Parquet"

In [0]:
# language_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/languages")

In [0]:
# %fs
# ls "abfss://silver@moviehistory22.dfs.core.windows.net/languages"

In [0]:
# display(spark.read.parquet(f"{silver_folder_path}/languages"))

In [0]:
# En todas las notebook creamos un archivo... ahora queremos crear las tablas.
# Entonces vamos a modificar la escritura de las notebooks que creamos para que guarde en tablas:
# Ahora se escribe en formato DELTA

language_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages")

In [0]:
%sql
SELECT * FROM movie_silver.languages

In [0]:
dbutils.notebook.exit("Success")